# NIK 2026 | Explainable Municipal Multi-Hazard Screening

**Reproducible research notebook — final corrected flood exposure and MHI**

This notebook uses the 355-municipality modelling set (excluding `1856` and `1144`) and keeps **one canonical calculation** of flood exposure, calibrated landslide probabilities and the relative multi-hazard index (MHI). Evaluation predictions and the final full-data mapping predictions are **different products** of the same model specification and are deliberately labelled separately.

**Execution:** Run **Restart Kernel → Run All** from `notebooks/` (or from the repository root). No cells depend on historical notebook state. Outputs are cleared in the GitHub copy. The notebook requires the QGIS-derived flood geometry CSV and the final municipality geometries already produced in the research project. It does *not* recreate their underlying GIS operations from source polygons.

**Scientific interpretation:** the fitted model describes municipality-level occurrence classification, not annual landslide probability. Flood exposure is mapped NVE flood caution-zone coverage (‰), not flood probability. MHI is a relative exploratory screening index, not a joint hazard probability.

## 1. Imports and project settings
All imports are in the single cell below. Install the project dependencies in the active notebook environment before running (pandas, numpy, geopandas, shapely, scikit-learn, matplotlib, SHAP, pyarrow, openpyxl only if otherwise needed). The standalone LaTeX table writer does not require Jinja2.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shap
import shapely
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 80)
pd.options.display.float_format = "{:.3f}".format

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"
INTERIM = DATA / "interim"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
IMAGES = ROOT / "images"
TABLES = ROOT / "tables"
for folder in (RESULTS, FIGURES, IMAGES, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

MODEL_PATH = INTERIM / "nik_merged_raw.parquet"
ORIGINAL_FLOOD_PATH = INTERIM / "flood_feature.parquet"
FLOOD_GEOMETRY_PATH = RESULTS / "flood_geometry_all.csv"
SSB_PATH = INTERIM / "ssb_municipality_areas_2026_clean.csv"
MUNICIPALITY_PATH = DATA / "municipalities_master.gpkg"
LAND_PATH = INTERIM / "municipalities_land_for_maps.gpkg"
SEA_PATH = INTERIM / "n50_havflate.gpkg"
EXCLUDED = {"1856", "1144"}
SEED = 42
VERSION = "v1"

required_files = [MODEL_PATH, ORIGINAL_FLOOD_PATH, FLOOD_GEOMETRY_PATH,
                  SSB_PATH, MUNICIPALITY_PATH]
missing_files = [str(p.relative_to(ROOT)) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError("Required project inputs are missing:\n" + "\n".join(missing_files))
if not LAND_PATH.exists() and not SEA_PATH.exists():
    raise FileNotFoundError(
        "Missing land-only map geometry and the N50 sea polygons needed to generate it."
    )
print("Project root:", ROOT)

### Shared helpers and validation
Municipality IDs stay as four-character strings. The checked merge helper fails on duplicate or incomplete keys rather than silently multiplying rows.

In [ ]:
def fix_ids(series):
    return (series.astype("string").str.strip()
            .str.replace(r"\.0$", "", regex=True).str.zfill(4))


def minmax(series):
    values = pd.to_numeric(series, errors="raise")
    spread = values.max() - values.min()
    if not np.isfinite(spread) or spread <= 0 or values.isna().any():
        raise ValueError("Cannot min–max normalize an empty, nonnumeric, missing or constant feature")
    return (values - values.min()) / spread


def metric_record(name, y_true, probabilities):
    # ROC-AUC is undefined if a spatial fold contains only one target class.
    y_arr = np.asarray(y_true)
    prob_arr = np.asarray(probabilities)
    return {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_arr, prob_arr) if len(np.unique(y_arr)) == 2 else np.nan,
        "Average Precision": average_precision_score(y_arr, prob_arr) if y_arr.sum() > 0 else np.nan,
        "Brier": brier_score_loss(y_arr, prob_arr),
    }


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as fp:
        for part in iter(lambda: fp.read(1 << 20), b""):
            h.update(part)
    return h.hexdigest()

## 2. Modelling dataset: loading, validation and EDA
The final feature dataset comes from `nik_merged_raw.parquet`, not from the earlier in-notebook exploratory join. No imputation is applied to the full data before splitting. Flood exposure is **excluded from all landslide predictors**.

In [ ]:
df_raw = pd.read_parquet(MODEL_PATH)
if not {"kommunenummer", "kommunenavn", "Yi_shallow"}.issubset(df_raw.columns):
    raise ValueError("Modelling file lacks the municipality ID, name or target")
df_raw["kommunenummer"] = fix_ids(df_raw["kommunenummer"])
if df_raw["kommunenummer"].duplicated().any():
    raise ValueError("Duplicate municipality codes in raw model file")

df_model = df_raw.loc[~df_raw["kommunenummer"].isin(EXCLUDED)].copy()
assert len(df_model) == 355 and df_model["kommunenummer"].nunique() == 355
assert df_model["Yi_shallow"].notna().all()
assert set(df_model["Yi_shallow"].unique()) == {0, 1}
print("Modelling municipalities:", len(df_model))
print("Class counts: ", df_model["Yi_shallow"].value_counts().to_string())
print("Missing columns: ", df_model.isna().sum().loc[lambda x: x > 0].to_string())
display(df_model[["kommunenummer", "kommunenavn", "Yi_shallow"]].head())

In [ ]:
share_features = [c for c in df_model.columns if c.startswith("share_")]
if share_features:
    share_summary = df_model[share_features].describe().T[["mean", "std", "min", "max"]]
    display(share_summary.round(3))
    # Exploratory overview only; this does not mutate the raw modelling features.
    df_model[share_features].hist(figsize=(14, 10), bins=25)
    plt.suptitle("Deposit-share feature distributions")
    plt.tight_layout()
    plt.show()

numeric = df_model.select_dtypes(include="number")
terrain = [c for c in ["elev_mean", "elev_stdev", "relief", "slp_mean", "slp_stdev", "slp_max"]
           if c in numeric.columns]
if terrain:
    display(numeric[terrain].describe().T.round(3))

missing = df_model.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

## 3. Random holdout evaluation
This reproduces the original 60% training / 20% validation / 20% test split with stratification and `random_state=42`. The validation portion is reserved; fitting here uses the **training subset only**. All imputation happens within each pipeline, avoiding pre-split imputation leakage. Final model selection and test reporting must be interpreted in light of the original experimental design.

In [ ]:
drop_cols = ["kommunenummer", "kommunenavn", "Yi_shallow",
             "flood_permille_clean", "geometry", "geometry_bbox",
             "slp_stdev", "elev_stdev"]
X = df_model.drop(columns=drop_cols, errors="ignore").copy()
for col in ["river_length_km", "river_density"]:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")
X = X.select_dtypes(include="number")
y = df_model["Yi_shallow"].astype(int)
assert X.index.equals(df_model.index)
assert "flood_permille_clean" not in X.columns
share_cols = [c for c in X.columns if c.startswith("share_")]
other_cols = [c for c in X.columns if c not in share_cols]
if not share_cols or not other_cols:
    raise ValueError("Unexpected feature schema: missing deposit shares or other predictors")
print("Predictors:", len(X.columns), X.columns.tolist())

idx_train_val, idx_test = train_test_split(
    df_model.index, test_size=0.2, stratify=y, random_state=SEED)
idx_train, idx_val = train_test_split(
    idx_train_val, test_size=0.25, stratify=y.loc[idx_train_val], random_state=SEED)
y_train, y_test = y.loc[idx_train], y.loc[idx_test]
print("Train / validation / test:", len(idx_train), len(idx_val), len(idx_test))

preprocessing = ColumnTransformer([
    ("shares", SimpleImputer(strategy="constant", fill_value=0), share_cols),
    ("other", SimpleImputer(strategy="median"), other_cols),
])

lr_pipeline = Pipeline([
    ("preprocessing", clone(preprocessing)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])
hgb_base = Pipeline([
    ("preprocessing", clone(preprocessing)),
    ("model", HistGradientBoostingClassifier(
        max_depth=5, learning_rate=0.01, max_iter=200,
        min_samples_leaf=10, l2_regularization=1, random_state=SEED)),
])
hgb_calibrated = CalibratedClassifierCV(
    estimator=hgb_base, method="sigmoid", cv=5)

In [ ]:
dummy_model = DummyClassifier(strategy="prior")
dummy_model.fit(np.zeros((len(idx_train), 1)), y_train)
dummy_prob = dummy_model.predict_proba(np.zeros((len(idx_test), 1)))[:, 1]

lr_pipeline.fit(X.loc[idx_train], y_train)
lr_prob = lr_pipeline.predict_proba(X.loc[idx_test])[:, 1]

hgb_calibrated.fit(X.loc[idx_train], y_train)
hgb_prob = hgb_calibrated.predict_proba(X.loc[idx_test])[:, 1]

holdout_scores = pd.DataFrame([
    metric_record("Dummy", y_test, dummy_prob),
    metric_record("Logistic Regression", y_test, lr_prob),
    metric_record("Calibrated HGB", y_test, hgb_prob),
])
display(holdout_scores.round(3))
holdout_scores.to_csv(RESULTS / "holdout_metrics.csv", index=False)

# Paper-reported values are *checks*, not inputs to the calculations.
paper_metrics = {"ROC-AUC": 0.815, "Average Precision": 0.974, "Brier": 0.075}
observed = holdout_scores.set_index("Model").loc["Calibrated HGB"]
for metric, expected in paper_metrics.items():
    delta = abs(float(observed[metric]) - expected)
    print(f"{metric}: recomputed={observed[metric]:.3f}, article={expected:.3f}, abs_difference={delta:.4f}")
    if delta >= 0.0015:
        print("  CHECK model inputs, library versions and split; do not silently copy article numbers.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, probs in [("Dummy", dummy_prob), ("Logistic Regression", lr_prob),
                    ("Calibrated HGB", hgb_prob)]:
    frac, mean = calibration_curve(y_test, probs, n_bins=8, strategy="quantile")
    ax.plot(mean, frac, marker="o", label=name)
ax.plot([0, 1], [0, 1], "--", color="0.5", label="Perfect calibration")
ax.set(xlabel="Mean predicted probability", ylabel="Observed positive fraction",
       title="Holdout reliability diagram")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "random_holdout_calibration.png", dpi=250)
plt.show()

### Stratified bootstrap intervals (on the held-out predictions)
Intervals reflect sampling variability **conditional on the fitted model/test split** and do not account for model retraining or spatial sampling uncertainty.

In [ ]:
y_true = y_test.to_numpy()
predictions = np.asarray(hgb_prob)
pos = np.flatnonzero(y_true == 1)
neg = np.flatnonzero(y_true == 0)
rng = np.random.default_rng(SEED)
bootstrap = []
for _ in range(3000):
    take = np.concatenate([
        rng.choice(pos, len(pos), replace=True),
        rng.choice(neg, len(neg), replace=True),
    ])
    record = metric_record("Calibrated HGB", y_true[take], predictions[take])
    bootstrap.append([record["ROC-AUC"], record["Average Precision"], record["Brier"]])
bootstrap = np.asarray(bootstrap)
for j, name in enumerate(["ROC-AUC", "Average Precision", "Brier"]):
    lo, hi = np.quantile(bootstrap[:, j], [0.025, 0.975])
    print(f"{name}: {observed[name]:.3f}; bootstrap 95% interval [{lo:.3f}, {hi:.3f}]")

## 4. Spatial block cross-validation
Five KMeans centroid-based spatial groups (EPSG:25833), evaluated with GroupKFold. Fold results and out-of-fold predictions are generated **within one CV loop** to avoid redundant fitting. The original fold grouping method and hyperparameters are preserved.

In [ ]:
municipalities_gdf = gpd.read_file(MUNICIPALITY_PATH)[
    ["kommunenummer", "kommunenavn", "geometry"]].copy()
municipalities_gdf["kommunenummer"] = fix_ids(municipalities_gdf["kommunenummer"])
assert len(municipalities_gdf) == 357
assert municipalities_gdf["kommunenummer"].is_unique
if municipalities_gdf.crs is None:
    raise ValueError("Municipalities GeoPackage must have a valid CRS")
municipalities_gdf = municipalities_gdf.to_crs(epsg=25833)

geo_cv = municipalities_gdf[["kommunenummer", "geometry"]].merge(
    df_model[["kommunenummer", "Yi_shallow"]], on="kommunenummer", how="inner", validate="one_to_one")
assert len(geo_cv) == 355
centers = geo_cv.geometry.centroid
coords = np.column_stack([centers.x, centers.y])
geo_cv["spatial_group"] = KMeans(n_clusters=5, random_state=SEED, n_init=10).fit_predict(coords)
group_lookup = geo_cv.set_index("kommunenummer")["spatial_group"]
groups = df_model["kommunenummer"].map(group_lookup)
assert groups.notna().all()
print("Spatial groups:")
display(geo_cv.groupby("spatial_group")["Yi_shallow"].agg(
    total="size", positive="sum", negative=lambda s: int((s == 0).sum())))

fig, ax = plt.subplots(figsize=(9, 11))
geo_cv.plot(column="spatial_group", categorical=True, cmap="tab10", legend=True,
            edgecolor="white", linewidth=0.2, ax=ax)
ax.set_title("Spatial cross-validation groups (KMeans, k=5)")
ax.set_axis_off()
fig.tight_layout()
fig.savefig(FIGURES / "spatial_cv_groups.png", dpi=300)
plt.show()

In [ ]:
spatial_results = []
oof = {name: np.full(len(X), np.nan) for name in
       ("Dummy", "Logistic Regression", "Calibrated HGB")}
cv = GroupKFold(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    fold_predictions = {}
    fold_predictions["Dummy"] = np.full(len(test_idx), y_tr.mean(), dtype=float)
    for name, blueprint in [("Logistic Regression", lr_pipeline),
                            ("Calibrated HGB", hgb_calibrated)]:
        model = clone(blueprint)
        model.fit(X_tr, y_tr)
        fold_predictions[name] = model.predict_proba(X_te)[:, 1]

    for name, probabilities in fold_predictions.items():
        oof[name][test_idx] = probabilities
        spatial_results.append({"Fold": fold, "Test size": len(test_idx),
                                "Positives": int(y_te.sum()),
                                "Negatives": int((y_te == 0).sum()),
                                **metric_record(name, y_te, probabilities)})
    print(f"Spatial fold {fold}/5 completed")

spatial_df = pd.DataFrame(spatial_results)
display(spatial_df.round(3))
display(spatial_df.groupby("Model")[["ROC-AUC", "Average Precision", "Brier"]]
        .agg(["mean", "std"]).round(3))
spatial_df.to_csv(RESULTS / "spatial_cv_results.csv", index=False)

for probabilities in oof.values():
    assert np.isfinite(probabilities).all()
oof_summary = pd.DataFrame([metric_record(name, y, values) for name, values in oof.items()])
display(oof_summary.round(3))
oof_summary.to_csv(RESULTS / "spatial_oof_summary.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name in ["Dummy", "Logistic Regression", "Calibrated HGB"]:
    part = spatial_df[spatial_df["Model"] == name].sort_values("Fold")
    ax.plot(part["Fold"], part["ROC-AUC"], marker="o", label=name)
ax.set(xlabel="Spatial validation fold", ylabel="ROC-AUC",
       title="Regional variation in model discrimination", ylim=(0, 1.05))
ax.grid(alpha=0.2)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "spatial_cv_roc_auc.png", dpi=300)
plt.show()

### Missing-terrain diagnostic and northern-block sensitivity
The analysis selects the block with the most missing `elev_mean` rather than assuming its numerical KMeans label. No causal interpretation is made about missing terrain or regional differences.

In [ ]:
terrain_check = df_model[["kommunenummer", "kommunenavn", "Yi_shallow",
                          "elev_mean", "relief", "slp_mean"]].copy()
terrain_check["spatial_group"] = groups.to_numpy()
missing_summary = terrain_check.groupby("spatial_group").agg(
    municipalities=("Yi_shallow", "size"),
    negatives=("Yi_shallow", lambda s: int((s == 0).sum())),
    missing_elevation=("elev_mean", lambda s: int(s.isna().sum())),
    missing_relief=("relief", lambda s: int(s.isna().sum())),
    missing_slope=("slp_mean", lambda s: int(s.isna().sum())))
display(missing_summary)

focus_group = missing_summary["missing_elevation"].idxmax()
focused = terrain_check.loc[terrain_check["spatial_group"] == focus_group].copy()
focused["missing_terrain"] = focused[["elev_mean", "relief", "slp_mean"]].isna().any(axis=1)
focused["oof_probability"] = oof["Calibrated HGB"][np.flatnonzero(groups.to_numpy() == focus_group)]
print("High-missingness spatial group:", focus_group)
display(pd.crosstab(focused["missing_terrain"], focused["Yi_shallow"], margins=True))
display(focused.groupby(["missing_terrain", "Yi_shallow"])["oof_probability"]
        .agg(["count", "mean", "min", "max"]).round(3))

In [ ]:
terrain_cols = ["elev_mean", "elev_stdev", "relief", "slp_mean", "slp_stdev", "slp_max"]
X_no_terrain = X.drop(columns=terrain_cols, errors="ignore")
nt_shares = [c for c in X_no_terrain if c.startswith("share_")]
nt_others = [c for c in X_no_terrain if c not in nt_shares]
nt_preprocess = ColumnTransformer([
    ("shares", SimpleImputer(strategy="constant", fill_value=0), nt_shares),
    ("other", SimpleImputer(strategy="median"), nt_others),
])
nt_base = Pipeline([
    ("preprocessing", nt_preprocess),
    ("model", clone(hgb_base.named_steps["model"])),
])
nt_model = CalibratedClassifierCV(estimator=nt_base, method="sigmoid", cv=5)
test_pos = np.flatnonzero(groups.to_numpy() == focus_group)
train_pos = np.flatnonzero(groups.to_numpy() != focus_group)
nt_model.fit(X_no_terrain.iloc[train_pos], y.iloc[train_pos])
nt_prob = nt_model.predict_proba(X_no_terrain.iloc[test_pos])[:, 1]
focus_scores = metric_record("Without terrain", y.iloc[test_pos], nt_prob)
original_focus = metric_record("Original HGB", y.iloc[test_pos], oof["Calibrated HGB"][test_pos])
display(pd.DataFrame([original_focus, focus_scores]).round(3))

## 5. Corrected flood exposure — one authoritative calculation
The numerator is `flood_land_m2` from the **QGIS-processed geometry file**, after overlapping caution polygons and mapped sea areas were handled upstream. The denominator is `ssb_area_m2` from the existing clean SSB input. The notebook audits the earlier exposure values but **never uses them in MHI**. The correction cannot be independently reconstructed from raw GIS polygons using these CSVs alone.

In [ ]:
ssb = pd.read_csv(SSB_PATH, dtype={"kommunenummer": "string"})
geometry = pd.read_csv(FLOOD_GEOMETRY_PATH, dtype={"kommunenummer": "string"})
old_flood = pd.read_parquet(ORIGINAL_FLOOD_PATH)
for frame in (ssb, geometry, old_flood):
    frame["kommunenummer"] = fix_ids(frame["kommunenummer"])

for name, frame in [("SSB", ssb), ("QGIS flood geometry", geometry),
                    ("Original flood", old_flood)]:
    if frame["kommunenummer"].duplicated().any():
        raise ValueError(f"{name}: duplicate municipality IDs")

names = municipalities_gdf[["kommunenummer", "kommunenavn"]].copy()
flood_final = (names.merge(
    ssb[["kommunenummer", "ssb_area_m2"]],
    on="kommunenummer", how="left", validate="one_to_one")
    .merge(geometry[["kommunenummer", "flood_total_m2", "flood_sea_m2",
                     "flood_land_m2", "sea_pct"]],
           on="kommunenummer", how="left", validate="one_to_one"))

for col in ["ssb_area_m2", "flood_total_m2", "flood_sea_m2", "flood_land_m2", "sea_pct"]:
    flood_final[col] = pd.to_numeric(flood_final[col], errors="raise")
assert len(flood_final) == 357 and flood_final["kommunenummer"].is_unique
assert flood_final[["ssb_area_m2", "flood_land_m2", "flood_total_m2"]].notna().all().all()
assert (flood_final["ssb_area_m2"] > 0).all()
assert (flood_final["flood_land_m2"] >= 0).all()
assert (flood_final["flood_land_m2"] <= flood_final["flood_total_m2"] + 1).all()

flood_final["flood_permille_final"] = (
    flood_final["flood_land_m2"] / flood_final["ssb_area_m2"] * 1000
)
assert flood_final["flood_permille_final"].between(0, 1000).all()
print("All municipalities:", len(flood_final))
print("Maximum flood exposure (‰):", round(flood_final["flood_permille_final"].max(), 3))
print("Missing corrected values:", flood_final["flood_permille_final"].isna().sum())
if not np.isclose(flood_final["flood_permille_final"].max(), 431.894, atol=0.015):
    print("WARNING: computed maximum does not match 431.894‰ in the revision notes; verify the source CSVs.")
display(flood_final.nlargest(10, "flood_permille_final")
        [["kommunenavn", "flood_permille_final"]].round(3))
flood_final.to_csv(RESULTS / "flood_exposure_corrected_2026.csv", index=False)

In [ ]:
# Audit only: compare original values with the corrected values on the SAME 355 municipalities.
old_values = old_flood[["kommunenummer", "flood_permille_clean"]].copy()
old_values["flood_permille_clean"] = pd.to_numeric(old_values["flood_permille_clean"], errors="coerce")
valid_codes = set(df_model["kommunenummer"])
flood_audit = flood_final.loc[flood_final["kommunenummer"].isin(valid_codes),
                              ["kommunenummer", "kommunenavn", "flood_permille_final"]].merge(
    old_values, on="kommunenummer", how="left", validate="one_to_one")
assert len(flood_audit) == 355
if flood_audit["flood_permille_clean"].isna().any():
    raise ValueError("Cannot audit old and corrected flood rankings: missing original values")
old_top = set(flood_audit.nlargest(10, "flood_permille_clean")["kommunenummer"])
corrected_top = set(flood_audit.nlargest(10, "flood_permille_final")["kommunenummer"])
print("Spearman old vs corrected flood:",
      round(flood_audit["flood_permille_clean"].corr(
          flood_audit["flood_permille_final"], method="spearman"), 3))
print("Old/corrected top-10 overlap:", len(old_top & corrected_top), "/ 10")
print("Corrected top-10:")
display(flood_audit.nlargest(10, "flood_permille_final")
        [["kommunenavn", "flood_permille_clean", "flood_permille_final"]].round(3))

## 6. Final 355-municipality map predictions
**Holdout evaluation above stays untouched.** Fit the final calibrated HGB using the same feature schema and all 355 municipalities, as in the final revision. These are fitted-data mapping scores and must not be presented as independent test predictions.

In [ ]:
final_hgb = clone(hgb_calibrated)
final_hgb.fit(X, y)
full_prob = final_hgb.predict_proba(X)[:, 1]
assert len(full_prob) == 355 and np.isfinite(full_prob).all()

landslide_final = df_model[["kommunenummer", "kommunenavn"]].copy()
landslide_final["landslide_prob_noflood"] = full_prob
assert landslide_final["kommunenummer"].is_unique
print("Final fitted-data landslide probability range:",
      round(landslide_final["landslide_prob_noflood"].min(), 4),
      round(landslide_final["landslide_prob_noflood"].max(), 4))

## 7. Corrected MHI and rankings — the only final version
The base 50/50 MHI uses **min–max normalization of landslide probability and corrected flood exposure across the same 355 modelled municipalities**. Municipality names come only from `df_model`; the flood lookup supplies only ID and value, avoiding `kommunenavn_x/y` merge errors.

In [ ]:
flood_lookup = flood_final[["kommunenummer", "flood_permille_final"]].copy()
df_final = landslide_final.merge(
    flood_lookup, on="kommunenummer", how="left", validate="one_to_one")
assert len(df_final) == 355 and df_final["kommunenummer"].nunique() == 355
assert df_final[["landslide_prob_noflood", "flood_permille_final"]].notna().all().all()

df_final["ls_prob_mm"] = minmax(df_final["landslide_prob_noflood"])
df_final["flood_mm"] = minmax(df_final["flood_permille_final"])
df_final["mhi"] = 0.5 * df_final["ls_prob_mm"] + 0.5 * df_final["flood_mm"]
assert df_final["mhi"].between(0, 1).all()

print("Final top 10 by MHI:")
display(df_final.nlargest(10, "mhi")[["kommunenavn", "landslide_prob_noflood",
                                           "flood_permille_final", "mhi"]].round(3))
print("Top 10 by flood exposure:")
display(df_final.nlargest(10, "flood_permille_final")
        [["kommunenavn", "flood_permille_final", "mhi"]].round(3))
print("Top 10 by landslide probability:")
display(df_final.nlargest(10, "landslide_prob_noflood")
        [["kommunenavn", "landslide_prob_noflood", "mhi"]].round(3))

df_final.to_csv(RESULTS / "mhi_pipeline_final_355.csv", index=False)

### Weight and normalization sensitivity
These are sensitivity checks on **the single corrected MHI dataset**, not alternative competing final MHI files. Top-10 overlap and rank correlations are computed, not hard-coded. Do not interpret MHI ranks as objectively stable risk ranks.

In [ ]:
df_final["mhi_25_75"] = 0.25 * df_final["ls_prob_mm"] + 0.75 * df_final["flood_mm"]
df_final["mhi_75_25"] = 0.75 * df_final["ls_prob_mm"] + 0.25 * df_final["flood_mm"]
df_final["ls_percentile"] = df_final["landslide_prob_noflood"].rank(pct=True)
df_final["flood_percentile"] = df_final["flood_permille_final"].rank(pct=True)
df_final["mhi_rank"] = 0.5 * df_final["ls_percentile"] + 0.5 * df_final["flood_percentile"]
main_top = set(df_final.nlargest(10, "mhi")["kommunenummer"])
for col in ["mhi_25_75", "mhi_75_25", "mhi_rank"]:
    alt_top = set(df_final.nlargest(10, col)["kommunenummer"])
    print(f"{col}: Spearman={df_final['mhi'].corr(df_final[col], method='spearman'):.3f}, "
          f"top-10 overlap={len(main_top & alt_top)}/10")

for col in ["mhi", "mhi_25_75", "mhi_75_25", "mhi_rank"]:
    df_final[f"rank_{col}"] = df_final[col].rank(ascending=False, method="min")
display(df_final.nlargest(10, "mhi")[["kommunenavn", "rank_mhi", "rank_mhi_25_75",
                                          "rank_mhi_75_25", "rank_mhi_rank"]].round(0))
print("MHI vs flood Spearman:", df_final["mhi"].corr(df_final["flood_permille_final"], method="spearman"))
print("MHI vs landslide Spearman:", df_final["mhi"].corr(df_final["landslide_prob_noflood"], method="spearman"))

# Resave the one authoritative 355-municipality CSV with sensitivity columns.
df_final.to_csv(RESULTS / "mhi_pipeline_final_355.csv", index=False)

## 8. SHAP — separately trained **uncalibrated** HGB
The SHAP explanation uses the *same base HGB hyperparameters and preprocessing* but is fit separately on the **training subset** without probability calibration. The contributions explain this uncalibrated model, **not** the calibrated ensemble's output. Global and local explanations are descriptive and should not be interpreted causally.

In [ ]:
shap_pipeline = clone(hgb_base)
shap_pipeline.fit(X.loc[idx_train], y_train)
shap_preprocess = shap_pipeline.named_steps["preprocessing"]
shap_tree = shap_pipeline.named_steps["model"]

train_transformed = shap_preprocess.transform(X.loc[idx_train])
test_transformed = shap_preprocess.transform(X.loc[idx_test])
raw_names = shap_preprocess.get_feature_names_out()
feature_names = [name.split("__", 1)[-1] for name in raw_names]
assert test_transformed.shape[1] == len(feature_names)
shap_train = pd.DataFrame(train_transformed, columns=feature_names, index=idx_train)
shap_test = pd.DataFrame(test_transformed, columns=feature_names, index=idx_test)

# Do not pass CalibratedClassifierCV to SHAP: explain the separately fit tree itself.
shap_explainer = shap.Explainer(shap_tree, shap_train, algorithm="tree")
shap_values = shap_explainer(shap_test, check_additivity=False)
shap.plots.bar(shap_values, max_display=10, show=True)
shap.plots.beeswarm(shap_values, max_display=10, show=True)
print("SHAP explained model: uncalibrated HistGradientBoostingClassifier")

In [ ]:
# One reproducible local explanation, aligned with the reserved test municipalities.
row_id = 0
print("Example test municipality:")
display(df_model.loc[[idx_test[row_id]], ["kommunenummer", "kommunenavn", "Yi_shallow"]])
shap.plots.waterfall(shap_values[row_id], max_display=10, show=True)
if "share_fluvial_flood" in feature_names:
    shap.plots.scatter(shap_values[:, "share_fluvial_flood"], show=True)

### Export municipality-level explanations for Streamlit

The interactive dashboard needs a `results/municipality_explanations_final.csv` file with one record for each of the 355 municipalities. These SHAP factors come from the **separately fitted uncalibrated HGB used in Section 8** (trained on the holdout training subset). They explain that model's raw score, **not** the calibrated probability from `final_hgb`. The original beeswarm and holdout results are unchanged. No text or feature importance is fabricated, and no old LLM summaries are reused.


In [ ]:
# Municipality-level SHAP feature contributions for the dashboard.
# Reuse the SAME uncalibrated HGB already fit in Section 8; do not refit or
# overwrite the calibrated model or the article's existing SHAP figure.
all_transformed = shap_preprocess.transform(X)
shap_all = pd.DataFrame(all_transformed, columns=feature_names, index=X.index)

# Explainer already comes from the separate uncalibrated HGB fitted above.
all_explanations = shap_explainer(shap_all, check_additivity=False)
all_shap_values = np.asarray(all_explanations.values)
if all_shap_values.shape != (len(X), len(feature_names)):
    raise ValueError(f"Unexpected SHAP shape: {all_shap_values.shape}")
if not np.isfinite(all_shap_values).all():
    raise ValueError("Nonfinite local SHAP values")

def describe_factors(values, positive=True, count=3):
    # SHAP values for the tree's output scale, not the calibrated probabilities.
    indexes = [i for i, value in enumerate(values)
               if (value > 1e-10 if positive else value < -1e-10)]
    indexes.sort(key=lambda i: abs(values[i]), reverse=True)
    if not indexes:
        return "None identified"
    return ", ".join(f"{feature_names[i]} ({values[i]:+.3f})"
                     for i in indexes[:count])

municipality_explanations = df_model[["kommunenummer", "kommunenavn"]].copy()
municipality_explanations["top_positive_shap"] = [
    describe_factors(v, positive=True) for v in all_shap_values]
municipality_explanations["top_negative_shap"] = [
    describe_factors(v, positive=False) for v in all_shap_values]
municipality_explanations["shap_model_provenance"] = (
    "Uncalibrated HGB trained on the holdout training subset; "
    "contributions are not calibrated-probability explanations"
)
assert len(municipality_explanations) == 355
assert municipality_explanations["kommunenummer"].is_unique
assert set(municipality_explanations["kommunenummer"]) == set(df_final["kommunenummer"])
assert municipality_explanations[["top_positive_shap", "top_negative_shap"]].notna().all().all()
explanations_path = RESULTS / "municipality_explanations_final.csv"
municipality_explanations.to_csv(explanations_path, index=False)
print("Saved:", explanations_path)
display(municipality_explanations.head())


## 9. Final land-only geometries and publication maps
The map geometry covers 357 municipalities. **Landslide Probability** and **MHI** are mapped for the 355 modelled municipalities (`1856` and `1144` are excluded); corrected **Flood Exposure** is mapped for all 357 municipalities.

The three maps match the layout of the manuscript-style reference figures. Flood exposure uses **fixed physical exposure classes** (0–50, 50–100, 100–200, 200–300, >300‰); Landslide and MHI use **five data-derived quintile classes**. These distinctions are essential: flood colours are **not** flood quintiles.

**Styling only:** missing or excluded municipalities are left white, not assigned a flood value, a probability, or zero risk. No grey polygon layer or grey legend swatch is drawn. All inputs, model predictions, and the MHI calculation remain unchanged.


In [ ]:
def municipality_area_km2(frame, code):
    found = frame.loc[frame["kommunenummer"] == code]
    if len(found) != 1:
        raise ValueError(f"Municipality {code} not found uniquely")
    return found.geometry.area.iloc[0] / 1e6

before_area = municipality_area_km2(municipalities_gdf, "1127")
if LAND_PATH.exists():
    land = gpd.read_file(LAND_PATH)[["kommunenummer", "geometry"]].copy()
    land["kommunenummer"] = fix_ids(land["kommunenummer"])
    land = land.to_crs(epsg=25833)
    if len(land) != 357 or municipality_area_km2(land, "1127") >= before_area * 0.99:
        raise ValueError("Cached land-only geometry appears incomplete or sea was not removed")
else:
    print("Generating missing land-only GeoPackage from N50 sea polygons...")
    sea = gpd.read_file(SEA_PATH)[["geometry"]].to_crs(epsg=25833)
    sea_index = sea.sindex
    cut_shapes = []
    for i, geom in enumerate(municipalities_gdf.geometry):
        nearby = sea_index.query(geom, predicate="intersects")
        if len(nearby) == 0:
            cut_shapes.append(geom)
        else:
            sea_union = shapely.union_all(sea.geometry.iloc[nearby].values)
            cut_shapes.append(geom.difference(sea_union))
        if (i + 1) % 25 == 0:
            print(f"Land geometry processed: {i + 1}/{len(municipalities_gdf)}")
    land = municipalities_gdf[["kommunenummer", "geometry"]].copy()
    land["geometry"] = cut_shapes
    if land.geometry.is_empty.any():
        raise ValueError("Land mask created an empty municipality geometry")
    land.to_file(LAND_PATH, driver="GPKG")

assert len(land) == 357 and land["kommunenummer"].is_unique
print("Randaberg geometry: before=", round(before_area, 2),
      "km²; after=", round(municipality_area_km2(land, "1127"), 2), "km²")

map_values = land.merge(df_final[["kommunenummer", "landslide_prob_noflood", "mhi"]],
                        on="kommunenummer", how="left", validate="one_to_one")
map_values = map_values.merge(flood_final[["kommunenummer", "flood_permille_final"]],
                              on="kommunenummer", how="left", validate="one_to_one")
gdf_maps = gpd.GeoDataFrame(map_values, geometry="geometry", crs=land.crs)
assert len(gdf_maps) == 357
assert gdf_maps["landslide_prob_noflood"].notna().sum() == 355
assert gdf_maps["mhi"].notna().sum() == 355
assert gdf_maps["flood_permille_final"].notna().sum() == 357

gpkg_path = INTERIM / "municipalities_land_pipeline_final.gpkg"
gdf_maps.to_file(gpkg_path, driver="GPKG")
print("Exported land-only mapping GeoPackage:", gpkg_path)

In [ ]:
# Publication-ready maps following the three supplied reference figures.
# All figures use only real values from flood_final or df_final.
# Excluded municipalities remain unfilled (white), never imputed.
MAP_SPECS = [
    {
        "variable": "landslide_prob_noflood",
        "title": "Modelled shallow-landslide occurrence probability\nby Norwegian municipality",
        "subtitle": None,
        "legend_title": "Relative probability\n(quintiles)",
        "filename": f"land_prob_pipeline_{VERSION}.png",
        "scheme": "quintiles",
        "colors": ["#eeebf7", "#c6c2e1", "#9890c6", "#6753a9", "#43158b"],
        "decimals": 4,
        "legend_position": "right",
        "show_scale": False,
        "source": (
            "Source: Calibrated HistGradientBoosting model; Kartverket municipal boundaries.\n"
            "355 municipalities modelled; 2 excluded (white gaps). Not annual event probability. EPSG:25833."
        ),
    },
    {
        "variable": "flood_permille_final",
        "title": "Municipal flood exposure in Norway",
        "subtitle": "NVE flood caution zones\nas a share of municipal land and inland water",
        "legend_title": "Flood exposure",
        "filename": f"flood_pipeline_{VERSION}.png",
        "scheme": "fixed_flood",
        "colors": ["#f0f6fc", "#c7deef", "#64abd4", "#2476b7", "#09295c"],
        "legend_position": "upper_left",
        "show_scale": True,
        "source": (
            "Sources: NVE flood caution areas, N50 sea polygons,\n"
            "SSB municipal areas, Kartverket boundaries.\n"
            "CRS: ETRS89 / UTM 33N (EPSG:25833)."
        ),
    },
    {
        "variable": "mhi",
        "title": "Relative multi-hazard screening index by municipality",
        "subtitle": (
            "Equal-weight index of modelled landslide probability\n"
            "and corrected municipal flood exposure"
        ),
        "legend_title": "Relative MHI (quintiles)",
        "filename": f"mhi_pipeline_{VERSION}.png",
        "scheme": "quintiles",
        "colors": ["#fff5e9", "#ffdaa7", "#ff8c60", "#dc3527", "#800000"],
        "decimals": 3,
        "legend_position": "upper_left",
        "show_scale": True,
        "source": (
            "Sources: Calibrated tree model, NVE flood caution areas,\n"
            "N50 sea polygons, SSB areas, Kartverket boundaries.\n"
            "CRS: ETRS89 / UTM 33N (EPSG:25833)."
        ),
    },
]


def add_true_scale_bar(ax, length_km=200):
    """Draw a real-distance scale bar in a metric projected CRS (EPSG:25833)."""
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    dx, dy = x1 - x0, y1 - y0
    length_m = length_km * 1000
    if dx <= length_m:
        raise ValueError("Map extent too narrow for requested scale bar")

    right = x1 - dx * 0.040
    left = right - length_m
    y = y0 + dy * 0.048
    ax.plot([left, right], [y, y], color="black", lw=6,
            solid_capstyle="butt", zorder=20)
    ax.text((left + right) / 2, y - dy * 0.015,
            f"{length_km} km", ha="center", va="top",
            fontsize=8.5, color="black", zorder=21)


def draw_publication_map(geo_frame, final_table, spec):
    var = spec["variable"]
    source = flood_final if var == "flood_permille_final" else final_table
    if geo_frame.crs is None or geo_frame.crs.to_epsg() != 25833:
        raise ValueError("Map scale bar and geometry require EPSG:25833")
    if len(geo_frame) != 357 or not geo_frame["kommunenummer"].is_unique:
        raise ValueError("Expected 357 unique municipality geometries")
    if not source["kommunenummer"].is_unique:
        raise ValueError(f"{var}: duplicated municipality IDs in source")

    # Select only source data for THIS layer, so no stale/suffixed values get used.
    values = source[["kommunenummer", var]].copy()
    values[var] = pd.to_numeric(values[var], errors="raise")
    plot_data = geo_frame[["kommunenummer", "geometry"]].merge(
        values, on="kommunenummer", how="left", validate="one_to_one"
    )
    n_expected = 357 if var == "flood_permille_final" else 355
    if plot_data[var].notna().sum() != n_expected:
        raise ValueError(f"{var}: expected {n_expected} valid municipalities")

    if spec["scheme"] == "fixed_flood":
        # The reference flood map uses absolute permille classes, NOT quintiles.
        boundaries = [0, 50, 100, 200, 300, np.inf]
        plot_data["_class"] = pd.cut(
            plot_data[var], bins=boundaries, labels=False, include_lowest=True
        )
        labels = ["0–50‰", "50–100‰", "100–200‰", "200–300‰", ">300‰"]
        # Represent open upper interval without Infinity in JSON manifest.
        manifest_boundaries = [0, 50, 100, 200, 300, None]
    else:
        boundaries = np.quantile(
            plot_data.loc[plot_data[var].notna(), var], np.linspace(0, 1, 6)
        )
        # Do not artificially split exact ties in SHAP/MHI quantiles.
        unique_edges = np.unique(boundaries)
        if len(unique_edges) != 6:
            raise ValueError(
                f"{var}: tied quantile thresholds; inspect before labelling five quintiles"
            )
        plot_data["_class"] = pd.cut(
            plot_data[var], bins=boundaries, labels=False, include_lowest=True
        )
        if var == "mhi":
            labels = ["Lowest 20%", "20–40%", "40–60%", "60–80%", "Highest 20%"]
        else:
            d = spec["decimals"]
            labels = [
                f"Q{i + 1}: {boundaries[i]:.{d}f}–{boundaries[i + 1]:.{d}f}"
                for i in range(5)
            ]
        manifest_boundaries = boundaries.tolist()

    visible = plot_data.loc[plot_data["_class"].notna()].copy()
    if len(visible) != n_expected or plot_data["_class"].isna().sum() != 357 - n_expected:
        raise AssertionError(f"{var}: class assignment/missing geometry mismatch")
    visible["_class"] = visible["_class"].astype(int)
    if not visible["_class"].between(0, 4).all():
        raise AssertionError(f"{var}: invalid colour class")

    colors = spec["colors"]
    cmap = ListedColormap(colors)
    if spec["legend_position"] == "right":
        # The paper reference places the purple map left and its legend to the right.
        fig = plt.figure(figsize=(9.2, 9.5), dpi=160, facecolor="white")
        ax = fig.add_axes([0.045, 0.10, 0.67, 0.79])
        legend_anchor = (0.755, 0.56)
    else:
        # Blue and orange maps fill the portrait page; legend floats at top left.
        fig = plt.figure(figsize=(8.2, 10.7), dpi=160, facecolor="white")
        ax = fig.add_axes([0.035, 0.065, 0.93, 0.86])
        legend_anchor = (0.060, 0.855 if var == "mhi" else 0.845)

    ax.set_facecolor("white")
    visible.plot(
        ax=ax, column="_class", categorical=True, cmap=cmap,
        vmin=-0.5, vmax=4.5,
        edgecolor="white", linewidth=0.15,
    )
    ax.set_aspect("equal")
    ax.set_axis_off()
    fig.suptitle(spec["title"], x=0.5, y=0.980,
                 fontsize=14 if var == "landslide_prob_noflood" else 13,
                 fontweight="semibold")
    if spec["subtitle"]:
        fig.text(0.035, 0.925, spec["subtitle"], ha="left", va="top",
                 fontsize=9.5, color="#202020", linespacing=1.35)

    # No grey legend entry and no background fill for excluded regions.
    legend_handles = [Patch(facecolor=c, edgecolor="none", label=label)
                      for c, label in zip(colors, labels)]
    legend = fig.legend(
        handles=legend_handles,
        title=spec["legend_title"],
        loc="upper left", bbox_to_anchor=legend_anchor,
        frameon=False, fontsize=9.3, title_fontsize=9.8,
        labelspacing=0.42, borderaxespad=0, handlelength=1.6,
        handleheight=0.8, handletextpad=0.65,
    )
    legend._legend_box.align = "left"
    if spec["show_scale"]:
        add_true_scale_bar(ax, length_km=200)
    fig.text(0.5, 0.015, spec["source"], ha="center", va="bottom",
             fontsize=7.2, color="#303030", linespacing=1.25)

    dest = IMAGES / spec["filename"]
    fig.savefig(dest, dpi=300, bbox_inches="tight", facecolor="white", pad_inches=0.10)
    plt.show()
    plt.close(fig)
    print(f"Saved: {dest} | displayed: {len(visible)}/{len(plot_data)} municipalities")
    return {"filename": str(dest.relative_to(ROOT)),
            "boundaries": manifest_boundaries,
            "classification": spec["scheme"]}


map_manifest = [draw_publication_map(gdf_maps, df_final, spec) for spec in MAP_SPECS]


## 10. Final Top 10, export manifest and consistency checks
The same `df_final` feeds the CSV, tables, maps and final manifest. LaTeX is written directly (no Jinja2 dependency), and saved files are associated with checksums for the paper and Streamlit.

In [ ]:
def latex_escape(text):
    replacements = {"&": r"\&", "%": r"\%", "_": r"\_", "#": r"\#",
                    "æ": r"{\ae}", "ø": r"{\o}", "å": r"{\aa}",
                    "Æ": r"{\AE}", "Ø": r"{\O}", "Å": r"{\AA}"}
    return "".join(replacements.get(char, char) for char in str(text))

top10 = df_final.nlargest(10, "mhi")[["kommunenummer", "kommunenavn",
                                        "landslide_prob_noflood", "flood_permille_final", "mhi"]].copy()
header = "\n".join([
    r"\begin{table}[t]", r"\centering",
    r"\caption{Top 10 municipalities by corrected relative multi-hazard index (equal weights).}",
    r"\label{tab:top_mhi}", r"\scriptsize", r"\begin{tabular}{r l c c c}",
    r"\toprule",
    "Rank & Municipality & Landslide prob. & Flood exposure (per mille) & MHI " + "\\\\",
    r"\midrule",
])
rows = [f"{rank} & {latex_escape(row['kommunenavn'])} & "
        f"{row['landslide_prob_noflood']:.3f} & {row['flood_permille_final']:.3f} & "
        f"{row['mhi']:.3f} \\\\"
        for rank, (_, row) in enumerate(top10.iterrows(), 1)]
footer = "\n".join([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
tex_path = TABLES / f"top_mhi_pipeline_{VERSION}.tex"
tex_path.write_text(header + "\n" + "\n".join(rows) + "\n" + footer + "\n", encoding="utf-8")
print("Saved:", tex_path)
display(top10.round(3))

In [ ]:
final_csv_path = RESULTS / "mhi_pipeline_final_355.csv"
assert final_csv_path.exists()
exported = pd.read_csv(final_csv_path, dtype={"kommunenummer": str})
exported["kommunenummer"] = fix_ids(exported["kommunenummer"])
assert len(exported) == 355
assert exported["kommunenummer"].is_unique
assert exported[["landslide_prob_noflood", "flood_permille_final", "mhi"]].notna().all().all()
assert np.allclose(exported["mhi"], 0.5 * minmax(exported["landslide_prob_noflood"])
                   + 0.5 * minmax(exported["flood_permille_final"]))
assert set(exported["kommunenummer"]) == set(df_model["kommunenummer"])

manifest = {
    "version": VERSION,
    "modelled_municipalities": len(df_final),
    "flood_exposure_municipalities": len(flood_final),
    "exclusions": sorted(EXCLUDED),
    "holdout_metrics_recomputed": holdout_scores.to_dict(orient="records"),
    "flood_max_permille_recomputed": float(flood_final["flood_permille_final"].max()),
    "model_csv": str(final_csv_path.relative_to(ROOT)),
    "model_csv_sha256": sha256(final_csv_path),
    "map_geometry": str(gpkg_path.relative_to(ROOT)),
    "map_geometry_sha256": sha256(gpkg_path),
    "flood_geometry_input_sha256": sha256(FLOOD_GEOMETRY_PATH),
    "ssb_area_input_sha256": sha256(SSB_PATH),
    "maps": map_manifest,
    "top10_mhi": top10.to_dict(orient="records"),
}
manifest_path = ROOT / f"nik_pipeline_{VERSION}_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("All final checks passed.")
print("Manifest:", manifest_path)
print("Final CSV:", final_csv_path)
print("Holdout validation and full-data fitted predictions are stored separately.")

---
**Before publishing on GitHub:** restart the kernel and run all cells with the original project inputs, inspect the printed metrics and top-10, compare exported maps with the supplied publication-style references, then save a clean output-cleared notebook. The code has not been executed against the private source data in this environment, so matching the reported numbers is still a verification step, not an established result. Do not commit `creds.json`, API keys, or other secrets.